# 实践 11.2  测序深度和分箱宽度对模拟结构检出的影响

《统计遗传与智能育种学》第十一章 · 电子资源 `11b_hic_depth_resolution.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：11b_hic_depth_resolution.ipynb。实践先构造固定的结构域边界和染色质环，只抽样一次高深度对称接触矩阵；中、低深度矩阵均从同一个高深度矩阵进行二项下采样，因此三者共享同一结构真值。随后聚合为原分箱宽度的1、2、4倍，比较零计数比例、预设边界的隔离分数、估计边界位置误差和染色质环相对同距离期望的富集，比较深度与分箱聚合设置下的检出结果。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果：高、中深度在三种聚合尺度下均在预设边界附近定位到分箱索引60；低深度细bin零比例约0.841，估计边界漂到72（误差12个原始bin），4倍聚合后边界误差降为4个原始bin。低深度细bin的环的同距离富集比反而升至约48.667，此时环像素只有4次接触，同距离均值约0.082。解读富集比还须考虑覆盖。

## 输入、完整代码、实际输出与解释

输入与可复现条件：120个原始分箱、预设边界索引60、预设环索引(35,82)；SEED=1102；中、低深度矩阵均由同一高深度矩阵下采样。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=1102
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1102


In [2]:
n=120; ii=np.arange(n); dist=np.abs(ii[:,None]-ii[None,:]); base=60/(dist+1)**.9; boundary=60; cross=(ii[:,None]<boundary)&(ii[None,:]>=boundary); base[cross|cross.T]*=.70; loop=(35,82); base[loop]+=80; base[loop[::-1]]+=80
def sample_symmetric(lam):
 upper=np.triu(rng.poisson(np.triu(lam))); return upper+np.triu(upper,1).T
def thin_symmetric(C,p):
 upper=np.triu(rng.binomial(np.triu(C),p)); return upper+np.triu(upper,1).T
high=sample_symmetric(base*1.5); matrices={'high':high,'medium':thin_symmetric(high,1/3),'low':thin_symmetric(high,.03)}
def aggregate_matrix(C,a):
 if a==1: return C
 k=n//a; return C.reshape(k,a,k,a).sum((1,3))
def metrics(C,a):
 A=aggregate_matrix(C,a); b=boundary//a; l=(loop[0]//a,loop[1]//a); w=max(2,6//a)
 def insulation(c):
  left=A[c-w:c,c-w:c]; right=A[c:c+w,c:c+w]; across=A[c-w:c,c:c+w]; return across.mean()/max((left.mean()+right.mean())/2,1e-8)
 candidates=range(max(w,b-16//a),min(len(A)-w,b+16//a+1)); est=min(candidates,key=insulation); error=abs(est*a-boundary)
 d=abs(l[1]-l[0]); vals=np.diag(A,k=d); enrich=A[l]/max(vals.mean(),1e-8)
 return np.mean(A==0),insulation(b),est*a,error,enrich,A.shape[0]
rows=[]
for depth,C in matrices.items():
 for a in [1,2,4]: rows.append((depth,a,*metrics(C,a)))
print(pd.DataFrame(rows,columns=['depth','bin_aggregation','zero_fraction','true_boundary_insulation','estimated_boundary_bin','boundary_error_bins','loop_enrichment','matrix_bins']).round(3).to_string(index=False))


 depth  bin_aggregation  zero_fraction  true_boundary_insulation  estimated_boundary_bin  boundary_error_bins  loop_enrichment  matrix_bins
  high                1          0.116                     0.267                      60                    0           33.948          120
  high                2          0.002                     0.267                      60                    0           11.250           60
  high                4          0.000                     0.253                      60                    0            3.821           30
medium                1          0.353                     0.259                      60                    0           34.913          120
medium                2          0.067                     0.259                      60                    0           11.739           60
medium                4          0.000                     0.246                      60                    0            3.782           30
   low              

结果解释：在结构真值完全相同的情况下，低深度细分箱矩阵84.1%的格子为0，边界估计从60漂移到72；本次聚合减少了零计数，4倍聚合后边界定位误差缩小。与此同时，低深度细分箱的同距离富集比反而升高，该比值受同距离背景及抽样波动影响，不能仅根据比值大小判断结构证据强弱。

可修改练习：把低深度矩阵的下采样比例从0.03改为0.01和0.10，比较zero_fraction、boundary_error_bins和loop_enrichment；说明为什么不能用单个富集比值定义有效分辨率。